# Notebook 04: Conversion, calibration and abstention threshold (E2), then freeze

**Framework component:** F4, conversion and calibration. Needs Notebook 03's 18 finished runs.

This notebook, in order:

1. applies the selection rule (training recipe) to the E1 results and records the selected architecture for each crop;
2. converts the selected architecture's three seed models to LiteRT **FP32, FP16 and full-integer INT8** (18 files), using 400 training images per crop as the INT8 representative set. The models output scores (logits); softmax and temperature are applied afterwards;
3. inspects each file (tensor types, quantisation scales and zero points, operators) and checks on 10 fixed images that Keras, FP32, FP16 and INT8 outputs are close and the top label agrees;
4. scores every file on the **selection** partition and applies the budgets: INT8 file at most 10 MiB per crop and 20 MiB for both, and an INT8 loss of at most 2 percentage points of accuracy and 0.02 macro-F1 relative to FP32;
5. fits the temperature T on the **temperature-calibration** partition and chooses the confidence threshold on the **threshold-selection** partition ;
6. **freezes** all choices in `frozen_choices_v2.json` with a SHA-256, before any internal or external test.

The internal test and the Ghanaian images are **not loaded** in this notebook.

**Runtime:** A100 or L4 (more CPU cores make the LiteRT scoring faster). About 45 to 75 minutes. Finished steps are skipped if you run all again.

In [ ]:
# 1. Setup and environment record
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, time, glob, tarfile, hashlib, datetime, platform, subprocess, gc, shutil
import numpy as np, pandas as pd
import tensorflow as tf, keras
from sklearn.metrics import f1_score, accuracy_score

DRIVE = '/content/drive/MyDrive/ocdf'
WORK = '/content/working'
PROTOCOL_VERSION = 'v2'
RUNS = f'{DRIVE}/runs'; OUT = f'{DRIVE}/records/nb04'; MODELS = f'{DRIVE}/models'
for d in [OUT, MODELS, WORK]: os.makedirs(d, exist_ok=True)
try: gpu_name = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
except Exception: gpu_name = 'none'
try:
    import ai_edge_litert; from ai_edge_litert.interpreter import Interpreter, OpResolverType; litert_pkg = f'ai_edge_litert {getattr(ai_edge_litert, "__version__", "")}'
except Exception:
    Interpreter = tf.lite.Interpreter; OpResolverType = tf.lite.experimental.OpResolverType; litert_pkg = 'tf.lite (bundled with TensorFlow)'
env = {'notebook': '04_conversion_calibration', 'run_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
       'python': sys.version.split()[0], 'tensorflow': tf.__version__, 'keras': keras.__version__, 'numpy': np.__version__,
       'litert_interpreter': litert_pkg, 'gpu': gpu_name, 'cpu_count': os.cpu_count(), 'platform': platform.platform()}
json.dump(env, open(f'{OUT}/environment_nb04.json', 'w'), indent=1); print(json.dumps(env, indent=1))

In [ ]:
# 2. Calibration, abstention and selection helpers (tested before delivery)
# ---- OCDF F4 calibration, abstention and selection helpers (Notebook 04) ----
import numpy as np

def softmax(z, T=1.0):
    z = np.asarray(z, dtype=np.float64) / T
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def nll(z, y, T=1.0):
    p = softmax(z, T)
    return float(-np.mean(np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1.0))))

def fit_temperature(z, y, lo=0.05, hi=20.0):
    """Calibration: positive T minimising negative log-likelihood on the calibration partition (Guo et al., 2017).
    Golden-section search on log T (NLL is unimodal in T for fixed logits)."""
    a, b = np.log(lo), np.log(hi); g = (np.sqrt(5) - 1) / 2
    c, d = b - g * (b - a), a + g * (b - a)
    fc, fd = nll(z, y, np.exp(c)), nll(z, y, np.exp(d))
    for _ in range(80):
        if fc < fd: b, d, fd = d, c, fc; c = b - g * (b - a); fc = nll(z, y, np.exp(c))
        else: a, c, fc = c, d, fd; d = a + g * (b - a); fd = nll(z, y, np.exp(d))
    T = float(np.exp((a + b) / 2))
    return {'T': round(T, 4), 'nll_before': round(nll(z, y, 1.0), 4), 'nll_after': round(nll(z, y, T), 4)}

def brier(p, y):
    oh = np.zeros_like(p); oh[np.arange(len(y)), y] = 1
    return float(np.mean(np.sum((p - oh) ** 2, axis=1)))

def ece(p, y, bins=15):
    conf = p.max(1); pred = p.argmax(1); acc = (pred == y).astype(float)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for i in range(bins):
        m = (conf > edges[i]) & (conf <= edges[i + 1]) if i else (conf >= edges[0]) & (conf <= edges[1])
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)

def threshold_table(p, y, groups, thresholds=None):
    """Coverage, selective accuracy and accepted groups for each threshold (see docs/METHODS.md)."""
    thresholds = np.round(np.arange(0.50, 0.995, 0.01), 2) if thresholds is None else thresholds
    conf = p.max(1); pred = p.argmax(1); groups = np.asarray(groups); rows = []
    for t in thresholds:
        acc_mask = conf >= t; n_acc = int(acc_mask.sum())
        rows.append({'threshold': float(t), 'coverage': n_acc / len(y),
                     'selective_accuracy': float((pred[acc_mask] == y[acc_mask]).mean()) if n_acc else None,
                     'selective_risk': float((pred[acc_mask] != y[acc_mask]).mean()) if n_acc else None,
                     'correct_accepted_yield': float(((pred == y) & acc_mask).mean()),
                     'accepted_groups': int(len(set(groups[acc_mask]))), 'accepted': n_acc})
    return rows

def choose_threshold(rows, min_acc=0.95, min_cov=0.50, min_groups=50, fallback=0.90):
    """Highest-coverage threshold with selective accuracy >= 0.95, coverage >= 0.50 and >= 50 accepted groups;
    otherwise 'target infeasible' and the predeclared 0.90 for description only."""
    ok = [r for r in rows if r['selective_accuracy'] is not None and r['selective_accuracy'] >= min_acc
          and r['coverage'] >= min_cov and r['accepted_groups'] >= min_groups]
    if ok:
        best = max(ok, key=lambda r: (r['coverage'], -r['threshold']))
        return {'threshold': best['threshold'], 'status': 'target met', **{k: best[k] for k in ('coverage', 'selective_accuracy', 'accepted_groups')}}
    fb = min(rows, key=lambda r: abs(r['threshold'] - fallback))
    return {'threshold': fallback, 'status': 'target infeasible (0.90 used for description only)',
            **{k: fb[k] for k in ('coverage', 'selective_accuracy', 'accepted_groups')}}

def per_class_coverage(p, y, t, classes):
    conf = p.max(1); out = {}
    for i, c in enumerate(classes):
        m = y == i
        out[c] = round(float((conf[m] >= t).mean()), 4) if m.any() else None
    return out

def select_architecture(agg_rows, size_bytes=None, margin=0.01):
    """Pre-declared selection rule: highest mean selection macro-F1; within `margin` of the best, the smaller converted file."""
    best = max(agg_rows, key=lambda r: r['mean_macro_f1'])
    close = [r for r in agg_rows if best['mean_macro_f1'] - r['mean_macro_f1'] <= margin + 1e-12]
    if len(close) > 1 and size_bytes:
        pick = min(close, key=lambda r: size_bytes[r['architecture']])
        return pick['architecture'], f'within {margin} of best; smallest FP32 file chosen'
    return best['architecture'], (f"highest mean selection macro-F1 ({best['mean_macro_f1']:.4f}); "
                                   f"no other architecture within {margin}")

In [ ]:
# 3. Frozen manifest, image archives, and the partitions this notebook may use
protocol = json.load(open(f'{DRIVE}/records/protocol_{PROTOCOL_VERSION}.json'))
mpath = f'{DRIVE}/manifests/{protocol["manifest"]}'
assert hashlib.sha256(open(mpath, 'rb').read()).hexdigest() == protocol['manifest_sha256'], 'manifest changed since it was frozen'
M = pd.read_csv(mpath, dtype=str)
ALLOWED = ['train', 'select', 'calib_T', 'thresh']                    # never 'test' or 'external' here
U = M[M.partition.isin(ALLOWED)].copy()
assert not U.partition.isin(['test', 'external']).any()
for src in ['D1', 'D4', 'D6']:
    if not os.path.isdir(f'{WORK}/{src}'):
        with tarfile.open(f'{DRIVE}/working/working_{src}.tar') as tar: tar.extractall(WORK, filter='data')
CLASSES = {'cassava': ['C01', 'C02', 'C03'], 'maize': ['M01', 'M02', 'M03', 'M04', 'M05']}
SEEDS = [17, 42, 2026]; DEPLOY_SEED = 42          # seed-42 models chosen in advance for interval estimates
IMG = 224
def load_img(path):                                  # identical to Notebook 03
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.image.resize_with_pad(img, IMG, IMG, method='bilinear')
    return tf.cast(tf.clip_by_value(tf.round(img), 0, 255), tf.uint8)
def load_array(df):
    ds = tf.data.Dataset.from_tensor_slices([f'{WORK}/{p}' for p in df.working_path]).map(load_img, num_parallel_calls=tf.data.AUTOTUNE).batch(256)
    return np.concatenate([b.numpy() for b in ds])
print(U.groupby(['crop', 'partition']).size().unstack())

In [ ]:
# 4. Architecture selection (pre-declared selection rule) from the E1 results
E1 = pd.DataFrame([json.load(open(p)) for p in glob.glob(f'{RUNS}/*/done.json')])
assert len(E1) == 18, f'expected 18 finished runs, found {len(E1)}'
SELECTED = {}
for crop in CLASSES:
    agg = (E1[E1.crop == crop].groupby('architecture').select_macro_f1.agg(['mean', 'std', 'count']).reset_index()
           .rename(columns={'mean': 'mean_macro_f1'}))
    arch, reason = select_architecture(agg.to_dict('records'))
    SELECTED[crop] = {'architecture': arch, 'reason': reason, 'e1': agg.round(4).to_dict('records')}
    print(crop, '->', arch, '|', reason)
json.dump(SELECTED, open(f'{OUT}/architecture_selection.json', 'w'), indent=1)

In [ ]:
# 5. E2 conversion: logits model, export, FP32 / FP16 / INT8, inspection and Keras parity
def logits_model(path):
    m = keras.models.load_model(path)
    dense = m.get_layer('probs')
    lin = keras.layers.Dense(dense.units, activation=None, name='logits')
    lm = keras.Model(m.inputs, lin(m.get_layer('dropout').output), name=m.name + '_logits')
    lin.set_weights(dense.get_weights())
    return m, lm

def make_interpreter(b, threads, xnnpack=True):
    if xnnpack:
        it = Interpreter(model_content=b, num_threads=threads); path = 'default (XNNPACK)'
    else:
        it = Interpreter(model_content=b, num_threads=threads, experimental_op_resolver_type=OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES)
        path = 'builtin kernels without XNNPACK'
    it.allocate_tensors(); return it, path

def _run(b, X, threads, batch, xnnpack):
    it, path = make_interpreter(b, threads, xnnpack)
    i, o = it.get_input_details()[0]['index'], it.get_output_details()[0]['index']
    it.resize_tensor_input(i, [batch, IMG, IMG, 3]); it.allocate_tensors()
    out = []
    for s in range(0, len(X), batch):
        xb = X[s:s + batch].astype(np.float32); n = len(xb)
        if n < batch: xb = np.concatenate([xb, np.zeros((batch - n, IMG, IMG, 3), np.float32)])
        it.set_tensor(i, xb); it.invoke(); out.append(it.get_tensor(o)[:n].copy())
    return np.concatenate(out), path

def run_tflite(b, X, threads=None, batch=32):
    # XNNPACK can fail at preparation or first invoke (seen for MobileNetV3-Small INT8 in E0): retry without it
    threads = threads or os.cpu_count()
    try: return _run(b, X, threads, batch, True)
    except Exception as e:
        print('  XNNPACK failed (', type(e).__name__, '); using builtin kernels'); return _run(b, X, threads, batch, False)

def inspect(b):
    try: it, path = make_interpreter(b, 1, True)
    except Exception: it, path = make_interpreter(b, 1, False)
    ins = [{'name': d['name'], 'dtype': str(d['dtype'].__name__), 'shape': d['shape'].tolist(),
            'quantization': [float(x) for x in d['quantization']]} for d in it.get_input_details()]
    outs = [{'name': d['name'], 'dtype': str(d['dtype'].__name__), 'shape': d['shape'].tolist(),
             'quantization': [float(x) for x in d['quantization']]} for d in it.get_output_details()]
    tens = it.get_tensor_details(); dt = pd.Series([t['dtype'].__name__ for t in tens]).value_counts().to_dict()
    try: ops = pd.Series([o['op_name'] for o in it._get_ops_details()]).value_counts().to_dict()
    except Exception as e: ops = {'unavailable': str(e)[:80]}
    float_ops = sum(v for k, v in ops.items() if k in ('DEQUANTIZE', 'QUANTIZE')) if isinstance(ops, dict) else None
    return {'inputs': ins, 'outputs': outs, 'tensor_dtypes': dt, 'operators': ops, 'quantize_dequantize_ops': float_ops,
            'interpreter_path': path}

def rep_set(crop, n=400):
    tr = U[(U.crop == crop) & (U.partition == 'train')]
    strata = tr.groupby(['research_label', 'source_id'])
    per = int(np.ceil(n / strata.ngroups))
    pick = pd.concat([g.sample(min(per, len(g)), random_state=2026) for _, g in strata]).sample(frac=1, random_state=2026).head(n)
    pick[['sample_id', 'research_label', 'source_id']].to_csv(f'{OUT}/representative_{crop}.csv', index=False)
    return load_array(pick)

CONV = []                       # one row per converted file
for crop in CLASSES:
    arch = SELECTED[crop]['architecture']; classes = CLASSES[crop]
    sel = U[(U.crop == crop) & (U.partition == 'select')]
    parity_df = sel.groupby('research_label').head(2).head(10)          # 10 fixed images, 2 per label where possible
    parity_df[['sample_id', 'research_label']].to_csv(f'{OUT}/parity_images_{crop}.csv', index=False)
    Xp = load_array(parity_df); R = None
    for seed in SEEDS:
        tag = f'{crop}_{arch}_s{seed}'; mdir = f'{MODELS}/{crop}'; os.makedirs(mdir, exist_ok=True)
        if all(os.path.exists(f'{mdir}/{tag}_{f}.tflite') for f in ['fp32', 'fp16', 'int8']) and os.path.exists(f'{OUT}/conv_{tag}.json'):
            CONV += json.load(open(f'{OUT}/conv_{tag}.json')); print('converted already:', tag); continue
        if R is None: R = rep_set(crop)
        keras.backend.clear_session()
        m, lm = logits_model(f'{RUNS}/{tag}/best_model.keras')
        k_prob = m.predict(Xp.astype(np.float32), verbose=0); k_log = lm.predict(Xp.astype(np.float32), verbose=0)
        assert np.allclose(softmax(k_log), k_prob, atol=1e-4), 'logits model does not reproduce the trained softmax'
        exp_dir = f'/content/export_{tag}'; shutil.rmtree(exp_dir, ignore_errors=True); lm.export(exp_dir, verbose=False)
        rows = []
        for fmt in ['fp32', 'fp16', 'int8']:
            c = tf.lite.TFLiteConverter.from_saved_model(exp_dir)
            if fmt == 'fp16':
                c.optimizations = [tf.lite.Optimize.DEFAULT]; c.target_spec.supported_types = [tf.float16]
            if fmt == 'int8':
                c.optimizations = [tf.lite.Optimize.DEFAULT]
                c.representative_dataset = lambda: ([R[j:j + 1].astype(np.float32)] for j in range(len(R)))
                c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
                c.inference_input_type = tf.float32; c.inference_output_type = tf.float32
            t0 = time.time(); b = c.convert(); secs = round(time.time() - t0, 1)
            path = f'{mdir}/{tag}_{fmt}.tflite'; open(path, 'wb').write(b)
            zp, ipath = run_tflite(b, Xp, threads=1, batch=1)
            rows.append({'crop': crop, 'architecture': arch, 'seed': seed, 'format': fmt, 'file': os.path.relpath(path, DRIVE),
                         'bytes': len(b), 'MiB': round(len(b) / 1048576, 3), 'sha256': hashlib.sha256(b).hexdigest(),
                         'convert_seconds': secs, 'inspection': inspect(b),
                         'parity_max_abs_prob_diff_vs_keras': float(np.abs(softmax(zp) - k_prob).max()),
                         'parity_top1_agreement_vs_keras': float((zp.argmax(1) == k_prob.argmax(1)).mean()),
                         'parity_images': len(Xp), 'interpreter_path': ipath})
            print(f'{tag} {fmt}: {rows[-1]["MiB"]} MiB, parity top-1 agreement {rows[-1]["parity_top1_agreement_vs_keras"]:.2f}, '
                  f'max prob diff {rows[-1]["parity_max_abs_prob_diff_vs_keras"]:.4f}, {ipath}')
        json.dump(rows, open(f'{OUT}/conv_{tag}.json', 'w'), indent=1, default=str); CONV += rows
        shutil.rmtree(exp_dir, ignore_errors=True); del m, lm; gc.collect()

In [ ]:
# 6. Score every file on selection, temperature-calibration and threshold partitions (logits saved to Drive)
LOGITS = f'{OUT}/logits'; os.makedirs(LOGITS, exist_ok=True)
for crop in CLASSES:
    arch = SELECTED[crop]['architecture']; classes = CLASSES[crop]
    X = {part: None for part in ['select', 'calib_T', 'thresh']}
    for seed in SEEDS:
        for fmt in ['fp32', 'fp16', 'int8']:
            tag = f'{crop}_{arch}_s{seed}_{fmt}'
            if all(os.path.exists(f'{LOGITS}/{tag}_{p}.npy') for p in X): continue
            b = open(f'{MODELS}/{crop}/{crop}_{arch}_s{seed}_{fmt}.tflite', 'rb').read()
            for part in X:
                if X[part] is None:
                    X[part] = load_array(U[(U.crop == crop) & (U.partition == part)].sort_values('sample_id'))
                t0 = time.time(); z, _ = run_tflite(b, X[part]); np.save(f'{LOGITS}/{tag}_{part}.npy', z)
                print(f'{tag} {part}: {len(z)} images in {time.time()-t0:.0f}s')
    del X; gc.collect()

In [ ]:
# 7. Budgets on the selection partition, temperature T (calib_T), threshold (thresh), then FREEZE
RES = []; THR_CURVES = []
for crop in CLASSES:
    arch = SELECTED[crop]['architecture']; classes = CLASSES[crop]
    parts = {p: U[(U.crop == crop) & (U.partition == p)].sort_values('sample_id') for p in ['select', 'calib_T', 'thresh']}
    ys = {p: np.array([classes.index(l) for l in d.research_label]) for p, d in parts.items()}
    for seed in SEEDS:
        for fmt in ['fp32', 'fp16', 'int8']:
            tag = f'{crop}_{arch}_s{seed}_{fmt}'
            z = {p: np.load(f'{LOGITS}/{tag}_{p}.npy') for p in parts}
            ps = softmax(z['select'])
            cal = fit_temperature(z['calib_T'], ys['calib_T'])
            pt = softmax(z['thresh'], cal['T'])
            curve = threshold_table(pt, ys['thresh'], parts['thresh'].group_id.values)
            for r in curve: THR_CURVES.append({'crop': crop, 'seed': seed, 'format': fmt, **r})
            ch = choose_threshold(curve)
            pc = softmax(z['calib_T'], cal['T'])
            RES.append({'crop': crop, 'architecture': arch, 'seed': seed, 'format': fmt,
                        'select_accuracy': accuracy_score(ys['select'], ps.argmax(1)),
                        'select_macro_f1': f1_score(ys['select'], ps.argmax(1), average='macro'),
                        'T': cal['T'], 'calib_nll_before': cal['nll_before'], 'calib_nll_after': cal['nll_after'],
                        'calib_ece_before': ece(softmax(z['calib_T']), ys['calib_T']), 'calib_ece_after': ece(pc, ys['calib_T']),
                        'threshold': ch['threshold'], 'threshold_status': ch['status'], 'thresh_coverage': ch['coverage'],
                        'thresh_selective_accuracy': ch['selective_accuracy'], 'thresh_accepted_groups': ch['accepted_groups'],
                        'thresh_coverage_per_class': per_class_coverage(pt, ys['thresh'], ch['threshold'], classes)})
R = pd.DataFrame(RES)
C = pd.DataFrame([{k: r[k] for k in ['crop', 'seed', 'format', 'bytes', 'MiB', 'sha256', 'file', 'parity_top1_agreement_vs_keras',
                                       'parity_max_abs_prob_diff_vs_keras', 'interpreter_path']} for r in CONV])
R = R.merge(C, on=['crop', 'seed', 'format'])
fp = R[R.format == 'fp32'].set_index(['crop', 'seed'])
R['acc_change_vs_fp32_pp'] = [100 * (r.select_accuracy - fp.loc[(r.crop, r.seed), 'select_accuracy']) for r in R.itertuples()]
R['macro_f1_change_vs_fp32'] = [r.select_macro_f1 - fp.loc[(r.crop, r.seed), 'select_macro_f1'] for r in R.itertuples()]
int8 = R[R.format == 'int8']
budget = {'int8_max_MiB_per_crop': 10, 'int8_max_MiB_both': 20, 'max_accuracy_loss_pp': 2, 'max_macro_f1_loss': 0.02, 'per_model': []}
for r in int8.itertuples():
    budget['per_model'].append({'crop': r.crop, 'seed': int(r.seed), 'MiB': float(r.MiB), 'size_ok': bool(r.MiB <= 10),
                                'accuracy_change_pp': round(float(r.acc_change_vs_fp32_pp), 3), 'accuracy_ok': bool(r.acc_change_vs_fp32_pp >= -2),
                                'macro_f1_change': round(float(r.macro_f1_change_vs_fp32), 4), 'macro_f1_ok': bool(r.macro_f1_change_vs_fp32 >= -0.02)})
dep = int8[int8.seed == DEPLOY_SEED]
budget['both_crops_int8_MiB_seed42'] = round(float(dep.MiB.sum()), 3); budget['both_ok_seed42'] = bool(dep.MiB.sum() <= 20)
budget['all_int8_within_budget'] = all(m['size_ok'] and m['accuracy_ok'] and m['macro_f1_ok'] for m in budget['per_model'])
print(R[['crop', 'seed', 'format', 'MiB', 'select_accuracy', 'select_macro_f1', 'acc_change_vs_fp32_pp', 'macro_f1_change_vs_fp32',
         'T', 'threshold', 'threshold_status', 'thresh_coverage', 'thresh_selective_accuracy']].round(4).to_string(index=False))
print(json.dumps(budget, indent=1, default=float))

# Format chosen for the deployed instance: INT8 if it passes every budget, otherwise the working FP16 alternative, else FP32
deploy = {}
for crop in CLASSES:
    ok = [m for m in budget['per_model'] if m['crop'] == crop and m['seed'] == DEPLOY_SEED][0]
    fmt = 'int8' if (ok['size_ok'] and ok['accuracy_ok'] and ok['macro_f1_ok']) else 'fp16'
    row = R[(R.crop == crop) & (R.seed == DEPLOY_SEED) & (R.format == fmt)].iloc[0]
    deploy[crop] = {'architecture': row.architecture, 'seed': DEPLOY_SEED, 'format': fmt, 'file': row.file, 'sha256': row.sha256,
                    'bytes': int(row.bytes), 'labels_in_output_order': CLASSES[crop], 'temperature_T': float(row['T']),
                    'threshold': float(row.threshold), 'threshold_status': row.threshold_status,
                    'input': 'float32 [1,224,224,3], RGB, values 0-255, aspect-preserving resize with zero padding (bilinear)',
                    'output': 'logits; probabilities = softmax(logits / T); abstain if max probability < threshold'}
frozen = {'protocol_version': PROTOCOL_VERSION, 'manifest_sha256': protocol['manifest_sha256'],
          'frozen_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'architecture_selection': SELECTED,
          'deployed_instance': deploy, 'all_variants': R.drop(columns=['thresh_coverage_per_class']).to_dict('records'),
          'note': 'Frozen before any internal-test or external (D3) image was loaded.', 'environment': env}
fpath = f'{DRIVE}/records/frozen_choices_{PROTOCOL_VERSION}.json'
if os.path.exists(fpath):
    print('A frozen file already exists; it is kept unchanged:', fpath)
else:
    json.dump(frozen, open(fpath, 'w'), indent=1, default=float)
fsha = hashlib.sha256(open(fpath, 'rb').read()).hexdigest()
open(f'{DRIVE}/records/frozen_choices_{PROTOCOL_VERSION}.sha256', 'w').write(fsha)
print('FROZEN:', fpath, fsha)
R.to_csv(f'{OUT}/e2_variants_selection.csv', index=False); pd.DataFrame(THR_CURVES).to_csv(f'{OUT}/threshold_curves.csv', index=False)
json.dump(budget, open(f'{OUT}/budget_check.json', 'w'), indent=1, default=float)
json.dump(CONV, open(f'{OUT}/conversion_inspection.json', 'w'), indent=1, default=str)

In [ ]:
# 8. Results file (no models): conversion inspection, budgets, temperatures, thresholds, frozen choices
import zipfile
res = '/content/nb04_results.zip'
with zipfile.ZipFile(res, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in [f'{OUT}/environment_nb04.json', f'{OUT}/architecture_selection.json', f'{OUT}/e2_variants_selection.csv',
              f'{OUT}/threshold_curves.csv', f'{OUT}/budget_check.json', f'{OUT}/conversion_inspection.json', fpath,
              f'{DRIVE}/records/frozen_choices_{PROTOCOL_VERSION}.sha256'] + glob.glob(f'{OUT}/representative_*.csv') + glob.glob(f'{OUT}/parity_images_*.csv'):
        if os.path.exists(p): z.write(p, os.path.basename(p))
from google.colab import files
files.download(res)

### When it finishes
The run records are written to Google Drive under `ocdf/records/` and also downloaded as `nb04_results.zip`. The copies used for the published results are in this repository under `records/nb04/`.